In [1]:
import pandas as pd
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Paragraph
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

def generate_report_cards(file_name):
    try:
        data = pd.read_excel(file_name)
        grouped = data.groupby(['Student ID', 'Name'])
        for (student_id, name), group in grouped:
            total_score = group['Score'].sum()
            average_score = group['Score'].mean()
            pdf_name = f"report_card_{student_id}.pdf"
            doc = SimpleDocTemplate(pdf_name)
            styles = getSampleStyleSheet()
            content = [
                Paragraph(f"Report Card for {name} (ID: {student_id})", styles['Title']),
                Paragraph(f"Total Score: {total_score}", styles['Normal']),
                Paragraph(f"Average Score: {average_score:.2f}", styles['Normal']),
            ]
            table_data = [['Subject', 'Score']] + group[['Subject', 'Score']].values.tolist()
            table = Table(table_data)
            table.setStyle(TableStyle([
                ('BACKGROUND', (0, 0), (-1, 0), colors.grey),
                ('GRID', (0, 0), (-1, -1), 1, colors.black),
                ('ALIGN', (0, 0), (-1, -1), 'CENTER')
            ]))
            content.append(table)
            doc.build(content)
            print(f"Generated: {pdf_name}")
    except Exception as e:
        print(f"Error: {e}")
generate_report_cards("student_scores.xlsx")


ModuleNotFoundError: No module named 'reportlab'

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
import time
import csv
def scrape_linkedin_data():
    options = Options()
    options.add_argument("--start-maximized")
    options.add_argument("--disable-notifications")
    service = Service("/path/to/chromedriver")
    driver = webdriver.Chrome(service=service, options=options)
    try:
        driver.get("https://www.linkedin.com/login")
        driver.find_element(By.ID, "username").send_keys("your_email@example.com")
        driver.find_element(By.ID, "password").send_keys("your_password")
        driver.find_element(By.ID, "password").send_keys(Keys.RETURN)
        time.sleep(5)
        search_query = "IIT graduate"
        driver.get(f"https://www.linkedin.com/search/results/people/?keywords={search_query}")
        time.sleep(5)
        with open("linkedin_data.csv", mode="w", newline="", encoding="utf-8") as file:
            writer = csv.writer(file)
            writer.writerow(["Name", "Job Title", "Company", "Industry"])
            for _ in range(5):
                profiles = driver.find_elements(By.CSS_SELECTOR, "div.entity-result__content")
                for profile in profiles:
                    try:
                        name = profile.find_element(By.CSS_SELECTOR, "span.entity-result__title-text").text
                        job_title = profile.find_element(By.CSS_SELECTOR, "div.entity-result__primary-subtitle").text
                        company = profile.find_element(By.CSS_SELECTOR, "div.entity-result__secondary-subtitle").text
                        industry = profile.find_element(By.CSS_SELECTOR, "div.entity-result__meta-info").text
                        writer.writerow([name, job_title, company, industry])
                    except Exception as e:
                        print(f"Could not extract a profile: {e}")
                driver.execute_script("window.scrollBy(0, 1000);")
                time.sleep(3)
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        driver.quit()
if __name__ == "__main__":
    scrape_linkedin_data()
